# Parallel Architecture — CrewAI

**Pattern:** Parallel (independent research tasks → aggregator)  
**Framework:** CrewAI  
**Task:** 3 city researchers (no dependencies) → aggregator ranks them

## Crew Structure

```
┌─────────────────────────────────────────────────────────────┐
│                          CREW                               │
│                                                             │
│  ┌──────────────┐  ┌──────────────┐  ┌────────────────────┐ │
│  │  Tokyo       │  │  Paris       │  │  Bangalore         │ │
│  │  Researcher  │  │  Researcher  │  │  Researcher        │ │
│  └──────┬───────┘  └──────┬───────┘  └─────────┬──────────┘ │
│         │                 │                     │            │
│  ┌──────▼───────┐  ┌──────▼───────┐  ┌──────────▼─────────┐ │
│  │  Task 1      │  │  Task 2      │  │  Task 3            │ │
│  │  context=[]  │  │  context=[]  │  │  context=[]        │ │
│  └──────────────┘  └──────────────┘  └────────────────────┘ │
│         │                 │                     │            │
│         └─────────────────┼─────────────────────┘            │
│                           ▼                                  │
│                  ┌────────────────┐                          │
│                  │  Aggregator    │                          │
│                  │  context=[all] │  output_pydantic         │
│                  └────────────────┘                          │
└─────────────────────────────────────────────────────────────┘
```

**Key:** Research tasks have `context=[]` — no dependencies on each other.  
The aggregator task has `context=[task1, task2, task3]` — sees all results.

In [ ]:
import os
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
gemini = LLM(model="gemini/gemini-2.0-flash", temperature=0)
print("✓ Ready")

In [ ]:
@tool("City Researcher")
def research_city(city: str) -> str:
    """Research weather, safety, and time for a city. Input: city name."""
    data = {
        "tokyo":     "Weather: Clear 18°C (9/10). Safety: Low (10/10). Time: 22:30 JST.",
        "paris":     "Weather: Partly Cloudy 16°C (7/10). Safety: Low (8/10). Time: 15:30 CET.",
        "bangalore": "Weather: Rainy 25°C (6/10). Safety: Medium (6/10). Time: 20:00 IST.",
    }
    return data.get(city.lower(), f"No data for '{city}'.")

class CityRank(BaseModel):
    city: str
    rank: int = Field(ge=1, le=10)
    reason: str

class RankedReport(BaseModel):
    rankings: List[CityRank]
    top_pick: str
    top_pick_reason: str

print("Tool + schema ready")

In [ ]:
# Three dedicated researchers — one per city
researcher_tokyo = Agent(
    role="Tokyo Researcher", goal="Research Tokyo travel data.",
    backstory="You specialize in Japan travel intelligence.",
    tools=[research_city], llm=gemini, verbose=False,
)
researcher_paris = Agent(
    role="Paris Researcher", goal="Research Paris travel data.",
    backstory="You specialize in European travel intelligence.",
    tools=[research_city], llm=gemini, verbose=False,
)
researcher_bangalore = Agent(
    role="Bangalore Researcher", goal="Research Bangalore travel data.",
    backstory="You specialize in South Asian travel intelligence.",
    tools=[research_city], llm=gemini, verbose=False,
)
aggregator = Agent(
    role="Travel Report Aggregator",
    goal="Rank all researched cities by weather and safety scores.",
    backstory="You synthesize multi-city research into ranked reports.",
    tools=[], llm=gemini, verbose=False,
)
print("4 agents: 3 researchers + 1 aggregator")

In [ ]:
# Independent research tasks (no context= on each other)
task_tokyo = Task(
    description="Research Tokyo: call research_city('Tokyo') and report weather, safety, time.",
    expected_output="Structured Tokyo data with scores.",
    agent=researcher_tokyo,
)
task_paris = Task(
    description="Research Paris: call research_city('Paris') and report weather, safety, time.",
    expected_output="Structured Paris data with scores.",
    agent=researcher_paris,
)
task_bangalore = Task(
    description="Research Bangalore: call research_city('Bangalore') and report weather, safety, time.",
    expected_output="Structured Bangalore data with scores.",
    agent=researcher_bangalore,
)
# Aggregator task receives ALL three research outputs
aggregate_task = Task(
    description="Rank Tokyo, Paris, Bangalore by combined weather + safety. Fill RankedReport.",
    expected_output="A RankedReport with all cities ranked.",
    agent=aggregator,
    context=[task_tokyo, task_paris, task_bangalore],  # ← sees all 3 research outputs
    output_pydantic=RankedReport,
)

crew = Crew(
    agents=[researcher_tokyo, researcher_paris, researcher_bangalore, aggregator],
    tasks=[task_tokyo, task_paris, task_bangalore, aggregate_task],
    process=Process.sequential,
    verbose=False,
)
print("Crew ready")

In [ ]:
result = crew.kickoff()
report: RankedReport = result.pydantic

if report:
    print("\n" + "="*50)
    print("RANKED REPORT")
    print("="*50)
    for r in sorted(report.rankings, key=lambda x: x.rank):
        print(f"  {r.rank}. {r.city}")
        print(f"     {r.reason}")
    print(f"\nTop Pick : {report.top_pick}")
    print(f"Why      : {report.top_pick_reason}")
else:
    print(result.raw)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `context=[]` on research tasks | No dependencies = parallel-capable in CrewAI |
| 3 dedicated researchers | Each agent has focused context + backstory for its city |
| `context=[t1, t2, t3]` on aggregator | Aggregator sees all parallel results before ranking |
| `output_pydantic=RankedReport` | Structured output even from multi-source synthesis |

### CrewAI Parallel vs LangGraph `Send()`
| | CrewAI | LangGraph |
|---|---|---|
| Parallelism | Independent tasks (no context) | `Send()` API with explicit fan-out |
| Agent identity | Named agents with backstories | Stateless node functions |
| Aggregation | `context=[all_tasks]` | `operator.add` reducer |
| Visualization | None built-in | `draw_mermaid_png()` |

### Next: [ADK Parallel →](../ADK/parallel.ipynb)